In [1]:
import boto3
print(f"Boto3 version: {boto3.__version__}")
usage_history = []

Boto3 version: 1.43.85


In [2]:
import boto3
import logging
import os
# logging.basicConfig(level=logging.DEBUG)
# boto3.set_stream_logger('', logging.DEBUG)

# Create logs directory if it doesn't exist
log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)

# Configure logging
logging.basicConfig(
    filename=os.path.join(log_dir, "log.txt"),
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# Log messages
logging.debug("This is a debug message")
logging.info("This is an info message")
logging.warning("This is a warning message")
logging.error("This is an error message")
logging.critical("This is a critical message")


region = "ap-south-1"
target_model_id = "anthropic.claude-haiku-4-5-20251001-v1:0"
# initialize a boto3 session with the specified region
session = boto3.Session(region_name=region)

bedrock = session.client("bedrock")
# list all inference profiles and filter for the one that matches the target model ID
profiles = bedrock.list_inference_profiles(
    typeEquals="SYSTEM_DEFINED"
)["inferenceProfileSummaries"]

matching_profiles = [
    profile
    for profile in profiles
    if any(
        target_model_id in model.get("modelArn", "")
        for model in profile.get("models", [])
    )
]

if not matching_profiles:
    raise RuntimeError(
        f"No inference profile for {target_model_id} is available in {region}."
    )

model_id = matching_profiles[0]["inferenceProfileId"]
print(f"Using inference profile: {model_id}")



Using inference profile: global.anthropic.claude-haiku-4-5-20251001-v1:0


In [3]:
client = session.client("bedrock-runtime")

In [4]:
def add_user_message(messages, text):

    user_message = {
        "role": "user",
        "content": text,
    }
    messages.append(user_message)
    print(messages)
    return user_message

def add_assistant_message(messages, text):

    assistant_message = {
        "role": "assistant",
        "content": text,
    }
    messages.append(assistant_message)
    return assistant_message

# def chat_with_model(messages):
#     system_prompt = """
#     You are an AWS cloud support specialist. Your job is to answer user queries related to AWS cloud hosting services.    
# """
#     response = client.converse(
#         modelId=model_id,
#         messages=messages,
#         system = [{"text": system_prompt}]
#     )
#     # return response["output"]["message"]["content"][0]["text"]
#     return response

# better version of chat_with_model that handles the system prompt and returns the full response
def chat_with_model(messages, system=None):
    params = {
        "modelId": model_id,
        "messages": messages}
    if system:
        params["system"] = [{"text": system}]
    response = client.converse(**params)
    return response["output"]["message"]["content"][0]["text"]


In [5]:
messages = []
add_user_message(messages, "write a code snippet that checks for duplicate chats in a string")
# Convert any plain-string user content to the Bedrock expected format
for msg in messages:
    if isinstance(msg.get("content"), str):
        msg["content"] = [{"text": msg["content"]}]

response = chat_with_model(messages, system="you are a python specialist and have deep coding knowledge and writes very concise code. the response should only contain the code and no comment")
print(response)

[{'role': 'user', 'content': 'write a code snippet that checks for duplicate chats in a string'}]
```python
def has_duplicate_chars(string: str) -> bool:
    return len(string) != len(set(string))

def find_duplicate_chars(string: str) -> set:
    return {char for char in string if string.count(char) > 1}

def first_duplicate_char(string: str) -> str | None:
    seen = set()
    for char in string:
        if char in seen:
            return char
        seen.add(char)
    return None

# Usage examples
print(has_duplicate_chars("hello"))  # True
print(has_duplicate_chars("abc"))    # False
print(find_duplicate_chars("programming"))  # {'g', 'r', 'm'}
print(first_duplicate_char("abccde"))  # 'c'
```


In [18]:
def has_duplicate_chats(s: str) -> bool:
    return len(s) != len(set(s))
has_duplicate_chats("hello")  # False

True

#### Temperatute: 

In [7]:
def chat_with_model(messages, system=None, Temperature=1 , max_tokens=1000):
    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": Temperature,
            "maxTokens": max_tokens
        }}
    if system:
        params["system"] = [{"text": system}]
    response = client.converse(**params)
    return response["output"]["message"]["content"][0]["text"]


In [8]:
messages = []

add_user_message(messages, "a movie idea in a single line")
# Convert any plain-string user content to the Bedrock expected format
for msg in messages:
    if isinstance(msg.get("content"), str):
        msg["content"] = [{"text": msg["content"]}]
print(messages)
text = chat_with_model(messages)
print(text)

[{'role': 'user', 'content': 'a movie idea in a single line'}]
[{'role': 'user', 'content': [{'text': 'a movie idea in a single line'}]}]
# A washed-up magician must pull off one final impossible trick to save his estranged daughter from the crime boss who's holding her hostage.


In [10]:
text = chat_with_model(messages, Temperature=1.0, max_tokens=500)
print(text)

# A time-traveling detective must solve her own murder before the killer erases her from history.


In [18]:
response = client.converse_stream(messages=messages, modelId=model_id)
# response['stream']
# for event in response["stream"]:
#     print(event)
text = ""
for event in response['stream']:
    if 'contentBlockDelta'in event:
        chunk = event['contentBlockDelta']['delta']['text']
        print(chunk)
        text+=chunk
        # print(" _next_ ")
print("\n\n Total Messages: \n" + text)


#
 A
 w
ashed
-up mag
ician discovers
 his stage
 ill
usions
 are bleeding
 into
 reality,
 and
 he
 must perform
 the
 ultimate trick to stop
 a
 parallel
 world
 from coll
iding
 with 
ours.


 Total Messages: 
# A washed-up magician discovers his stage illusions are bleeding into reality, and he must perform the ultimate trick to stop a parallel world from colliding with ours.


TypeError: string indices must be integers